In [ ]:
# Libraries.
import numpy as np
import pandas as pd

In [ ]:
# Dataset containing the TPM values for each sample and gene.
humanDataRaw = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.parquet")
display(humanDataRaw.head())

,Description,GTEX-1117F-0005-SM-HL9SH,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-1117F-0011-R11b-SM-GIN8R,GTEX-1117F-0011-R2b-SM-GI4VL,GTEX-1117F-0011-R3a-SM-GJ3PJ,GTEX-1117F-0011-R4b-SM-GI4VM,GTEX-1117F-0011-R5a-SM-GI4VW,GTEX-1117F-0011-R6a-SM-GI4VX,GTEX-1117F-0011-R7a-SM-H65ZK,...,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2326-SM-GOQYU,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2526-SM-GOQZ3,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
Name,,,,,,,,,,,,,,,,,,,,,
ENSG00000290825.2,DDX11L16,0.000000,0.000000,0.015825,0.000000,0.000000,0.034443,0.000000,0.013671,0.000000,...,0.028982,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016663,0.0000,0.000000
ENSG00000223972.6,DDX11L1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000
ENSG00000310526.1,WASH7P,1.163903,1.260954,3.023480,0.335984,0.631406,0.844382,1.409834,1.508202,1.038700,...,4.863250,2.082317,1.840574,3.879934,1.826294,3.870932,1.475226,6.310629,0.7036,2.185337
ENSG00000243485.6,MIR1302-2HG,0.000000,0.000000,0.027433,0.000000,0.027257,0.000000,0.000000,0.023698,0.021193,...,0.000000,0.000000,0.000000,0.059820,0.000000,0.000000,0.000000,0.000000,0.0000,0.042076
ENSG00000237613.3,FAM138A,0.000000,0.023562,0.000000,0.012465,0.034029,0.037269,0.000000,0.000000,0.013229,...,0.000000,0.000000,0.021915,0.000000,0.000000,0.041812,0.000000,0.018030,0.0000,0.026265


In [ ]:
# Metadata for each sample, contains information on the tissue type.
humanMetaData = pd.read_csv("https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt", sep="\t")
display(humanMetaData.head())

/var/folders/w9/_9xtsnhn18g346dc_nzw211c0000gn/T/ipykernel_11856/2999871758.py:1: DtypeWarning: Columns (0: SMGTC) have mixed types. Specify dtype option on import or set low_memory=False.
  humanMetaData = pd.read_csv("https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt", sep="\t")


,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,...,SMSHRTRT,SMSMRDHQ,SMSMRTHQ,SMPRERDHQ,SMPRERTHQ,SMSMGNDT,SMPREGNDT,SMRDLNMN,SMRDLNMD,SMRDLNSD
0,BMS-X4LF-0126-SM-4JBHL,NaN,B1,NaN,7.5,Thyroid,Thyroid,UBERON:0002046,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BMS-X4LF-0226-SM-4JBJ3,NaN,B1,NaN,6.9,Blood Vessel,Artery - Pulmonary,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BMS-X4LF-0326-SM-4JBIR,NaN,B1,NaN,7.4,Muscle,Muscle - Skeletal,UBERON:0011907,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BMS-X4LF-0426-SM-4JBIS,NaN,B1,NaN,7.1,Skin,Skin - Sun Exposed (Lower leg),UBERON:0004264,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BMS-X4LF-0526-SM-4JBHX,NaN,B1,NaN,8.8,Adrenal Gland,Adrenal Gland,UBERON:0002369,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
humanDataProcessed = []
tissueTypes = humanMetaData["SMTS"].unique()
for tissueType in tissueTypes:
    # Extract samples that are expressed in tissueType.
    samplesWithTissue = humanMetaData[humanMetaData["SMTS"] == tissueType]
    sampleIDs = samplesWithTissue["SAMPID"]

    # Extract the TPMs value from humanDataRaw.
    expressionLevels = [humanDataRaw[sampleID].to_numpy()for sampleID in sampleIDs if sampleID in humanDataRaw.columns]

    # Turn the TPM array into a dataframe and format the rows and columns.
    sampleDF = pd.DataFrame(expressionLevels)
    sampleDF.columns = humanDataRaw.index.to_numpy()
    sampleDF.index = [sampleID for sampleID in sampleIDs if sampleID in humanDataRaw.columns]
    sampleDF["Tissue.Type"] = tissueType
    
    humanDataProcessed.append(sampleDF)
    print(f"Completed {tissueType.title()} Dataset.")

humanData = pd.concat(humanDataProcessed)
humanData.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanData.parquet")

Completed Thyroid Dataset.


In [ ]:
humanData.isna().sum().sum()